# DevContext AI — RAG Layer Checkpoint
## Problem Statement
End-to-end test: ingest a sample repo + docs + schema, answer 3 onboarding queries with source citations, pass the pipeline health check.

## My Approach
* Each source type has a fundamentally different structure, so I matched the chunking strategy to that structure rather than applying one-size-fits-all splitting. 
* For code, AST extraction gives clean function-level units — no character boundary will ever cut across a function signature or its docstring. 
* For docs, RecursiveCharacterTextSplitter with a 300-char window and 50-char overlap handles unstructured prose without losing cross-sentence context. 
* For SQL schema, sqlglot gives real semantic awareness — I chunk at both table and column granularity, so a query about a specific column can retrieve exactly that chunk rather than swimming through an entire CREATE TABLE block.
* On the embedding side, I deliberately used two models to show I understand the per-collection strategy. BAAI/bge-base-en-v1.5 for code (strong retrieval model, HF Inference API supported), all-mpnet-base-v2 for docs and schema (excellent sentence semantics). 
* Both happen to be 768-dim, but I still created 3 separate ChromaDB collections and registered the correct embedding function on each - because dimension alone doesn't guarantee semantic alignment. The embedding function registered on a collection determines what model ChromaDB uses at query time, so mismatching it would silently corrupt results even if dimensions matched.

## What I Learned
* The most non-obvious thing was how ChromaDB handles the relationship between pre-computed embeddings and the registered embedding function. When you call collection.add(embeddings=...), ChromaDB stores those vectors as-is. But when you call collection.query(query_embeddings=...), it uses the collection's registered embedding function to embed the query - not the model you used at ingest time. 

    * So if you register hf_ef_mpnet on the code collection but pass BGE-produced embeddings at ingest, every query will compare a MPNet-encoded query against BGE-encoded documents. The cosine similarity scores look plausible (they don't error out), but the retrieval quality is garbage. This is a silent failure — the kind that makes RAG "work" in demos but fail in production.

* I also learned that HF Inference API returns 3-D tensors for some models (shape n_texts × seq_len × dim) because it returns token-level embeddings rather than pooled sentence embeddings. You have to mean-pool along axis 1 yourself, otherwise storing the raw output causes shape mismatches when ChromaDB tries to do cosine similarity.

## Where I Got Stuck
* Two things slowed me down. First, the embedding function mismatch on the code collection — I initially created collection_code with hf_ef_mpnet before realising the queries were comparing the wrong vector spaces. The fix was trivial (create hf_ef_bge and pass it to the code collection), but it took a few wrong retrieval results before I traced it back to the collection registration rather than the embedding step.

* Second, I didn't account for idempotency on re-runs. ChromaDB's get_or_create_collection returns the existing collection if it already exists, but calling .add() again on an already-populated collection throws a duplicate-ID error. Adding a safe_add() guard (if collection.count() == 0) fixed this cleanly.

## What I'd Do Differently
* In a real codebase, I'd separate ingestion from retrieval into two distinct scripts or modules rather than running them in the same notebook. Ingestion is a one-time operation per source version; retrieval is called on every query. Mixing them means you re-embed and re-ingest every time you restart the notebook, which is slow and error-prone.

* I'd also use nomic-embed-text-v1.5 for docs in production - it has an 8k context window versus MPNet's 512 tokens, which matters when onboarding docs have long sections that don't split cleanly. And for SQL schema I'd switch to BAAI/bge-m3, which is multilingual and handles mixed SQL dialects better. The sprint models (all-mpnet-base-v2 + bge-base-en-v1.5) were the right pragmatic choice for getting the checkpoint working on the HF free tier, but I'd flag both upgrades in a production design doc.

## My Solution (Naive)
*First attempt — written before reviewing feedback*

In [1]:
# -- sample_code.py --
samplecode_py="""
def authenticate_user(username, password):
    \"\"\"Validates credentials against the users table. Returns JWT on success.\"\"\"
    ...

def process_payment(order_id, amount, method):
    \"\"\"Charges the given method, updates orders table, emits payment_completed event.\"\"\"
    ...

def send_welcome_email(user_id):
    \"\"\"Fetches user from DB, renders Jinja template, sends via SendGrid.\"\"\"
    ...
"""

# -- onboarding_guide.md --
onboarding_guide_md="""
## Auth System
Authentication uses JWT tokens. The `authenticate_user` function validates credentials stored in the `users` table. Tokens expire after 24 hours.
New engineers should never store tokens in localStorage — use httpOnly cookies.

## Payments
All payments route through Stripe. The `process_payment` function is the single entrypoint — never call Stripe directly from other services.
"""

# -- schema.sql --
schema_sql="""
CREATE TABLE users (
    id SERIAL PRIMARY KEY,
    username VARCHAR(100) UNIQUE NOT NULL,
    password_hash TEXT NOT NULL,
    created_at TIMESTAMP DEFAULT NOW()
);

CREATE TABLE orders (
    id SERIAL PRIMARY KEY,
    user_id INT REFERENCES users(id),
    amount DECIMAL(10,2),
    status VARCHAR(50),
    created_at TIMESTAMP DEFAULT NOW()
);
"""

In [1]:
Queries=[
"how does authentication work and where should tokens be stored?"
,"what does the orders table contain?"
,"what happens when process_payment is called?"
]

In [ ]:
import os
from dotenv import load_dotenv
import requests
import numpy as np

#chunking import
import ast
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sqlglot import parse, exp

#embedding import
from huggingface_hub import login

#LLm import
from groq import Groq

import chromadb
import chromadb.utils.embedding_functions as embedding_functions

load_dotenv()

HF_TOKEN = os.getenv("HUGGING_FACE_API_KEY")
model_ids = {
    "MPNet_doc_sql": "sentence-transformers/all-mpnet-base-v2"
    ,"baai_code":"BAAI/bge-base-en-v1.5"
}

# Create the persistent client
client_chromadb = chromadb.PersistentClient(path="./chroma_db")

client_groq = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "llama-3.1-8b-instant" 

d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#Chunking 
#For doc - recursive_splitter
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
recursive_chunks = recursive_splitter.split_text(onboarding_guide_md)

In [ ]:
#for code & schema - AST
def ast_chunking(code: str) -> list:

    tree_code=ast.parse(code)
    code_chunks=[]

    for node in ast.iter_child_nodes(tree_code) :
        if isinstance(node , ast.FunctionDef ) :
            segment=ast.get_source_segment(code , node )
            if segment :
                code_chunks.append(segment)
        elif isinstance(node , ast.ClassDef):
            for child_node in ast.iter_child_nodes(node) :
                if isinstance(child_node , ast.FunctionDef ) :
                    segment=ast.get_source_segment(code , child_node )
                    if segment :
                        code_chunks.append(segment)
        
    return code_chunks

In [6]:
#Code chunking
code_chunks=ast_chunking(samplecode_py)

In [7]:
def chunk_sql_schema(sql: str) ->list:
    """
    Chunk SQL schema into semantic units:    - one chunk per CREATE TABLE - optional smaller chunks per column
    """
    parsed = parse(sql)
    chunks = []

    for stmt in parsed:
        # Chunk full statement
        chunks.append({
            "type": "table",
            "name": stmt.find(exp.Table).name,
            "content": stmt.sql(pretty=True)
        })

        # Chunk columns inside CREATE TABLE
        schema = stmt.find(exp.Schema)
        if schema:
            table_name = stmt.find(exp.Table).name

            for col in schema.expressions:
                chunks.append({
                    "type": "column",
                    "table": table_name,
                    "content": col.sql(pretty=True)
                })

    return chunks

In [8]:
#schema_chunks
schema_chunks = chunk_sql_schema(schema_sql)

for i, chunk in enumerate(schema_chunks, 1):
    print(f"\n--- CHUNK {i} ---")
    print(chunk)



--- CHUNK 1 ---
{'type': 'table', 'name': 'users', 'content': 'CREATE TABLE users (\n  id SERIAL PRIMARY KEY,\n  username VARCHAR(100) UNIQUE NOT NULL,\n  password_hash TEXT NOT NULL,\n  created_at TIMESTAMP DEFAULT NOW()\n)'}

--- CHUNK 2 ---
{'type': 'column', 'table': 'users', 'content': 'id SERIAL PRIMARY KEY'}

--- CHUNK 3 ---
{'type': 'column', 'table': 'users', 'content': 'username VARCHAR(100) UNIQUE NOT NULL'}

--- CHUNK 4 ---
{'type': 'column', 'table': 'users', 'content': 'password_hash TEXT NOT NULL'}

--- CHUNK 5 ---
{'type': 'column', 'table': 'users', 'content': 'created_at TIMESTAMP DEFAULT NOW()'}

--- CHUNK 6 ---
{'type': 'table', 'name': 'orders', 'content': 'CREATE TABLE orders (\n  id SERIAL PRIMARY KEY,\n  user_id INT REFERENCES users (\n    id\n  ),\n  amount DECIMAL(10, 2),\n  status VARCHAR(50),\n  created_at TIMESTAMP DEFAULT NOW()\n)'}

--- CHUNK 7 ---
{'type': 'column', 'table': 'orders', 'content': 'id SERIAL PRIMARY KEY'}

--- CHUNK 8 ---
{'type': 'column

In [3]:
#Embedding
def get_api_embeddings(texts, model_id):
    api_url = f"https://router.huggingface.co/hf-inference/models/{model_id}/pipeline/feature-extraction"

    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    
    payload = {
        "inputs": texts,
        "options": {"wait_for_model": True} 
    }
    
    response = requests.post(api_url, headers=headers, json=payload)
    
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text}")
        return None
    
    result = response.json()
    arr= np.array(result)
    if arr.ndim == 3:  # mean pool token embeddings
        arr = arr.mean(axis=1)
    return arr 

In [18]:
#Embed
code_embed=get_api_embeddings(code_chunks , model_ids["baai_code"] )
doc_embed = get_api_embeddings(recursive_chunks, model_ids["MPNet_doc_sql"])

sql_strings = [chunk['content'] for chunk in schema_chunks]
schema_embed = get_api_embeddings(sql_strings, model_ids["MPNet_doc_sql"])

In [ ]:
#chromadb
# Register the correct HF embedding function per collection so
# query-time embeddings always match ingest-time embeddings.

hf_ef_mpnet = embedding_functions.HuggingFaceEmbeddingFunction(
    api_key=HF_TOKEN, model_name=model_ids["MPNet_doc_sql"]
)
hf_ef_bge = embedding_functions.HuggingFaceEmbeddingFunction(
    api_key=HF_TOKEN, model_name=model_ids["baai_code"]
)

collection_doc    = client_chromadb.get_or_create_collection("docbase_devcontext",    embedding_function=hf_ef_mpnet)
collection_schema = client_chromadb.get_or_create_collection("schemabase_devcontext", embedding_function=hf_ef_mpnet)
collection_code   = client_chromadb.get_or_create_collection("codebase_devcontext",   embedding_function=hf_ef_bge)

# ── Ingest (skip if already populated) ───────────────────────────────────────
def safe_add(collection, embeddings, documents, ids, metadatas):
    """Add only if the collection is empty (idempotent re-runs)."""
    if collection.count() == 0:
        collection.add(embeddings=embeddings.tolist(), documents=documents, ids=ids, metadatas=metadatas)
        # print(f"{collection.name}: {collection.count()} chunks ingested")
    # else:
        # print(f"{collection.name}: already has {collection.count()} chunks, skipping add")

safe_add(
    collection_doc,
    doc_embed, recursive_chunks,
    [f"doc_{i}" for i in range(len(recursive_chunks))],
    [{"source": "doc", "file_name": "onboarding_guide_md", "d_id": i} for i in range(len(recursive_chunks))]
)

safe_add(
    collection_schema,
    schema_embed, sql_strings,
    [f"schema_{i}" for i in range(len(schema_chunks))],
    [{"source": "schema", "file_name": "schema_sql", "d_id": i} for i in range(len(schema_chunks))]
)

safe_add(
    collection_code,
    code_embed, code_chunks,
    [f"code_{i}" for i in range(len(code_chunks))],
    [{"source": "code", "file_name": "samplecode_py", "d_id": i} for i in range(len(code_chunks))]
)


In [22]:
def get_answer_from_context(context, user_question):
    prompt = f"""
    You are a technical assistant. Use the provided context to answer the question.
    
    CONTEXT:
    {context}
    
    USER QUESTION:
    {user_question}
    
    INSTRUCTIONS:
    1. Answer in 2-3 concise sentences.
    2. Merge information from different files into a single coherent answer.
    3. At the very end of your answer, list the unique filenames used in brackets, like this: [Source: file1, file2]
    4. Start your response with "Answer: "
    """
    
    response = client_groq.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1 # Lower temperature for higher accuracy in citations
    )
    return response.choices[0].message.content

In [23]:
def get_context(chromadb_result_Set) -> str:
    context = ""
    for i in range(len(chromadb_result_Set['documents'][0])):
        file_name = chromadb_result_Set['metadatas'][0][i]['file_name']
        content = chromadb_result_Set['documents'][0][i]
        context += f"[{file_name}]: {content}\n"
    return context

In [ ]:
def search_ans(query_text:str) :    
# 1. Generate BOTH embeddings (1 hit each)
    query_vec_mpnet = get_api_embeddings(query_text, model_ids["MPNet_doc_sql"])
    query_vec_bge = get_api_embeddings(query_text, model_ids["baai_code"])
    
    # 2. Vector Search (using correct vectors for each model)
    res_doc = collection_doc.query(embeddings=[query_vec_mpnet], n_results=3)
    res_sql = collection_schema.query(embeddings=[query_vec_mpnet], n_results=3)
    res_code = collection_code.query(embeddings=[query_vec_bge], n_results=3)

    context = "\n".join([
        get_context(res_doc),
        get_context(res_sql),
        get_context(res_code),
    ]).strip()
    
    return get_answer_from_context(context , query_text)

In [ ]:
#test search function
search_ans(Queries[0])

'Answer: Authentication uses JWT tokens, which are validated by the `authenticate_user` function against the `users` table. Tokens expire after 24 hours and should be stored in httpOnly cookies, not in localStorage. This ensures secure token storage and prevents unauthorized access.\n\n[Source: onboarding_guide_md, samplecode_py]'

In [26]:
for query in Queries:
    print(f''' Question : {query} ''') 
    print(f'''{search_ans(query)}''')
    print()

 Question : how does authentication work and where should tokens be stored? 
Answer: Authentication uses JWT tokens, which are validated by the `authenticate_user` function against the `users` table. Tokens expire after 24 hours and should be stored in httpOnly cookies, not in localStorage. This ensures secure token management and prevents unauthorized access.

[Source: onboarding_guide_md, samplecode_py]

 Question : what does the orders table contain? 
Answer: The orders table contains information about each order, including the user who made the order (user_id), the amount of the order (amount), the status of the order (status), and the timestamp when the order was created (created_at). The user_id is a foreign key referencing the id of the user in the users table. This information is used to track and manage orders in the system. [Source: schema_sql, onboarding_guide_md]

 Question : what happens when process_payment is called? 
Answer: When `process_payment` is called, it charges 